In [ ]:
!pip install -q monai
!pip install -q wandb

In [ ]:
%%writefile /kaggle/working/train.py

import os
import shutil

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_download

from monai.transforms import (
    Compose,
    RandFlipd,
    RandRotate90d,
    RandAffined,
    RandGaussianNoised,
    RandAdjustContrastd,
    RandGaussianSmoothd,
)
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.networks.utils import one_hot
from monai.inferers import sliding_window_inference

from accelerate import Accelerator

In [ ]:
%%writefile -a /kaggle/working/train.py

CHANNELS = [1, 32, 64, 128, 256, 512]
NUM_LABELS = 5
PATCH_SIZE = (96, 96, 96)
LR = 1e-4
BATCH_SIZE = 8
EPOCHS = 50

DATASET_REPO = "AG2307/pengwin-2026-anatomy-segmentation"
HF_REPO = "AG2307/pengwin2026-anatomy-segmentation-model"

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
LOCAL_CHECKPOINT = f"{CHECKPOINT_DIR}/best_model.pth"

In [ ]:
%%writefile -a /kaggle/working/train.py


class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv3d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class Unet(nn.Module):
    def __init__(self, channels, num_labels):
        super().__init__()
        self.encoders = nn.ModuleList(
            [DoubleConv(channels[i], channels[i + 1]) for i in range(len(channels) - 1)]
        )
        self.decoders = nn.ModuleList(
            [
                DoubleConv(channels[i], channels[i - 1])
                for i in range(len(channels) - 1, 1, -1)
            ]
        )
        self.reduce = nn.ModuleList(
            [
                nn.Conv3d(
                    in_channels=channels[i],
                    out_channels=channels[i - 1],
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,
                )
                for i in range(len(channels) - 1, 1, -1)
            ]
        )
        self.pool = nn.MaxPool3d(2)
        self.upsample = nn.Upsample(
            scale_factor=2,
            mode="trilinear",
            align_corners=True,
        )
        self.final_conv = nn.Conv3d(
            in_channels=channels[1],
            out_channels=num_labels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=True,
        )

    def forward(self, x):
        skips = []
        for enc in self.encoders[:-1]:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.encoders[-1](x)

        skips = list(reversed(skips))
        for i, dec in enumerate(self.decoders):
            x = self.upsample(x)
            x = self.reduce[i](x)
            x = torch.cat([x, skips[i]], dim=1)
            x = dec(x)
        x = self.final_conv(x)
        return x

In [ ]:
%%writefile -a /kaggle/working/train.py


class PelvicDataset(Dataset):
    def __init__(self, hf_dataset, transforms=None):
        self.dataset = hf_dataset
        self.transforms = transforms

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]
        if self.transforms:
            sample = self.transforms(sample)
        return sample


def build_transforms():
    return Compose(
        [
            RandAffined(
                keys=["image", "label"],
                prob=0.3,
                rotate_range=(0.15, 0.15, 0.15),
                scale_range=(0.1, 0.1, 0.1),
                translate_range=(10, 10, 5),
                mode=("bilinear", "nearest"),
                padding_mode="border",
            ),
            RandGaussianNoised(keys=["image"], prob=0.2, mean=0.0, std=0.1),
            RandAdjustContrastd(keys=["image"], prob=0.3, gamma=(0.7, 1.3)),
            RandGaussianSmoothd(keys=["image"], prob=0.2, sigma_x=(0.5, 1.0)),
        ]
    )


def build_dataloaders(accelerator):
    with accelerator.main_process_first():
        data = load_dataset(DATASET_REPO)
        data = data.remove_columns(
            ["patient_id", "original_spacing", "original_shape", "original_affine"]
        )
        data = data.with_format("torch")

    train_dataset = PelvicDataset(data["train"], transforms=build_transforms())
    val_dataset = PelvicDataset(data["validation"], transforms=None)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    return train_loader, val_loader

In [ ]:
%%writefile -a /kaggle/working/train.py


def maybe_download_checkpoint(accelerator, api, hf_token):
    """Main process pulls the latest checkpoint from HF if it exists."""
    if accelerator.is_main_process:
        # remove any stale local checkpoint from a previous session
        if os.path.exists(LOCAL_CHECKPOINT):
            os.remove(LOCAL_CHECKPOINT)
            accelerator.print("Removed stale local checkpoint.")

        try:
            downloaded = hf_hub_download(
                repo_id=HF_REPO,
                filename="best_model.pth",
                repo_type="model",
                token=hf_token,
            )
            shutil.copy(downloaded, LOCAL_CHECKPOINT)
            accelerator.print("Downloaded checkpoint from HuggingFace.")
        except Exception as e:
            accelerator.print(f"No remote checkpoint found: {e}")
    accelerator.wait_for_everyone()


def load_checkpoint(accelerator, model, optimizer, scheduler):
    """Restore state from a local checkpoint if present. Returns (start_epoch, best_dice)."""
    if not os.path.exists(LOCAL_CHECKPOINT):
        accelerator.print("No checkpoint found, starting fresh...")
        return 1, 0.0

    accelerator.print("Found existing checkpoint, loading...")
    ckpt = torch.load(LOCAL_CHECKPOINT, map_location="cpu")
    accelerator.unwrap_model(model).load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_dice = ckpt["best_dice"]
    accelerator.print(
        f"  Resuming from epoch {start_epoch}, best Dice: {best_dice:.4f}"
    )
    return start_epoch, best_dice


def save_checkpoint(
    accelerator, model, optimizer, scheduler, epoch, best_dice, api, hf_token
):
    """Save locally and push to HF (main process only — caller must guard)."""
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": accelerator.unwrap_model(model).state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_dice": best_dice,
        },
        LOCAL_CHECKPOINT,
    )
    accelerator.print(f"  New best! Dice: {best_dice:.4f}")

    api.upload_file(
        path_or_fileobj=LOCAL_CHECKPOINT,
        path_in_repo="best_model.pth",
        repo_id=HF_REPO,
        repo_type="model",
        token=hf_token,
        commit_message=f"Epoch {epoch} | Dice: {best_dice:.4f}",
    )
    accelerator.print("  Pushed to HuggingFace.")

In [ ]:
%%writefile -a /kaggle/working/train.py


def train_one_epoch(accelerator, model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0
    num_batches = len(loader)

    pbar = tqdm(loader, desc="Training", disable=not accelerator.is_main_process)
    for i, batch in enumerate(pbar):
        with accelerator.accumulate(model):
            pred = model(batch["image"])
            loss = loss_fn(pred, batch["label"])
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item()
        if i % 5 == 0:
            pbar.set_postfix(
                loss=f"{loss.item():.4f}", avg_loss=f"{total_loss / (i + 1):.4f}"
            )

    return total_loss / num_batches


@torch.no_grad()
def evaluate(accelerator, model, loader, loss_fn):
    """Runs on the main process only (caller guards). Returns (val_loss, val_dice)."""
    dice_metric = DiceMetric(include_background=False, reduction="mean")
    unwrapped = accelerator.unwrap_model(model)
    unwrapped.eval()

    total_loss = 0.0
    num_batches = len(loader)

    accelerator.print(num_batches)

    pbar = tqdm(loader, desc="Validation", disable=not accelerator.is_main_process)
    for i, batch in enumerate(pbar):
        image = batch["image"].to(accelerator.device)
        label = batch["label"].to(accelerator.device)

        pred = sliding_window_inference(
            inputs=image,
            roi_size=PATCH_SIZE,
            sw_batch_size=2,
            predictor=unwrapped,
            overlap=0.25,
            mode="gaussian",
        )

        total_loss += loss_fn(pred, label).item()

        pred_hard = torch.argmax(pred, dim=1, keepdim=True)
        dice_metric(
            one_hot(pred_hard, num_classes=NUM_LABELS),
            one_hot(label, num_classes=NUM_LABELS),
        )

        if i % 5 == 0:
            pbar.set_postfix(loss=f"{total_loss / (i + 1):.4f}")

    val_loss = total_loss / num_batches
    val_dice = dice_metric.aggregate().item()
    dice_metric.reset()
    return val_loss, val_dice

In [ ]:
%%writefile -a /kaggle/working/train.py


def build_model_and_optim():
    model = Unet(channels=CHANNELS, num_labels=NUM_LABELS)
    loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )
    return model, loss_fn, optimizer, scheduler

In [ ]:
%%writefile -a /kaggle/working/train.py


def main(gradient_accumulation_steps=2, mixed_precision="fp16"):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    hf_token = os.environ.get("HF_TOKEN")

    accelerator = Accelerator(
        gradient_accumulation_steps=gradient_accumulation_steps,
        mixed_precision=mixed_precision,
        log_with="wandb",
        project_dir=CHECKPOINT_DIR,
    )

    accelerator.init_trackers(
        "pengwin2026",
        config={
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "patch_size": PATCH_SIZE[0],
            "channels": CHANNELS,
            "num_classes": NUM_LABELS,
        },
    )

    # HF API + repo
    api = HfApi()
    if accelerator.is_main_process:
        api.create_repo(
            repo_id=HF_REPO,
            repo_type="model",
            private=True,
            token=hf_token,
            exist_ok=True,
        )

    # data, model, prepare
    train_loader, val_loader = build_dataloaders(accelerator)
    model, loss_fn, optimizer, scheduler = build_model_and_optim()
    model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
        model, optimizer, train_loader, val_loader, scheduler
    )
    accelerator.register_for_checkpointing(scheduler)

    # resume
    maybe_download_checkpoint(accelerator, api, hf_token)
    start_epoch, best_dice = load_checkpoint(accelerator, model, optimizer, scheduler)

    # training
    for epoch in range(start_epoch, EPOCHS + 1):
        accelerator.print(f"Epoch {epoch}")
        train_loss = train_one_epoch(
            accelerator, model, train_loader, optimizer, loss_fn
        )
        val_loss, val_dice = evaluate(accelerator, model, val_loader, loss_fn)

        if accelerator.is_main_process:
            scheduler.step(val_loss)

            accelerator.print(
                f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}"
            )
            accelerator.log(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "val_dice": val_dice,
                    "lr": optimizer.param_groups[0]["lr"],
                }
            )

            if val_dice > best_dice:
                best_dice = val_dice
                save_checkpoint(
                    accelerator,
                    model,
                    optimizer,
                    scheduler,
                    epoch,
                    best_dice,
                    api,
                    hf_token,
                )

        accelerator.wait_for_everyone()

    accelerator.end_training()


if __name__ == "__main__":
    main()

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("hf_auth_token")
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

!accelerate launch --num_processes=2 --mixed_precision=fp16 /kaggle/working/train.py